# Download FLUX.2 and NVARC to a Modal Volume

This tutorial downloads the checkpoint artifacts pinned by Diffusion-ARC directly from Hugging Face and Kaggle into persistent Modal storage. It does not use Google Drive or `rclone`.

**Audience:** Diffusion-ARC development in the Modal Notebook web interface.

**Prerequisites:**

- A Modal Notebook configured with a CPU kernel. A GPU is not required for downloading or verification.
- A Modal Volume named `diffusion-arc-models`, attached at `/mnt/diffusion-arc-models`.
- A Modal Secret attached to the notebook with the keys `HF_TOKEN` and `KAGGLE_API_TOKEN`.
- Accepted access conditions for [`black-forest-labs/FLUX.2-dev`](https://huggingface.co/black-forest-labs/FLUX.2-dev).
- About 32 GB of persistent storage plus time to download and hash the files.

**Learning goals:**

- Download the exact FLUX and NVARC sources currently used by Diffusion-ARC.
- Keep the checkpoints across notebook kernel restarts.
- Fail early on incomplete or incompatible artifacts.
- Produce a credential-free reproducibility manifest.

Tokens are read from the notebook environment. They are never written to the Volume, notebook cells, outputs, or manifest.


## Outline

1. Attach the Modal Volume and Secrets.
2. Install download-only dependencies.
3. Validate the persistent mount and declare pinned sources.
4. Download FLUX, the standalone VAE, and NVARC.
5. Validate structure, sizes, and project checkpoint hashes.
6. Write a manifest and verify persistence.


## 1. One-time Modal and account preparation

In the Modal Notebook sidebar, complete these steps before running any code:

1. Open the filesystem panel, create or attach a Volume named `diffusion-arc-models`, and confirm that it appears at `/mnt/diffusion-arc-models`.
2. Use a CPU kernel for this notebook. Keeping a GPU attached during a long download wastes GPU credit.
3. Create a Modal Secret containing both `HF_TOKEN` and `KAGGLE_API_TOKEN`, then attach that Secret to this notebook.
4. For `HF_TOKEN`, create a read token and accept the gated [`FLUX.2-dev`](https://huggingface.co/black-forest-labs/FLUX.2-dev) conditions with the same account.
5. For `KAGGLE_API_TOKEN`, generate the current token under [Kaggle API settings](https://www.kaggle.com/settings/api). Use the token value, not a legacy `kaggle.json` document.

The public FLUX Base repository and the published NVARC model may sometimes download anonymously, but this notebook requires both tokens so every source, including the gated standalone VAE, follows one deterministic path.


## 2. Install download-only dependencies

This cell installs only Hugging Face Hub and KaggleHub. It deliberately does not replace PyTorch, Diffusers, Transformers, or CUDA packages in the Modal kernel. Rerun it after a fresh kernel start.


In [ ]:
from __future__ import annotations

import importlib.metadata
import subprocess
import sys

PINNED_DOWNLOAD_PACKAGES = {
    "huggingface-hub": "0.36.2",
    "kagglehub": "1.0.2",
}

requirements = []
for distribution, expected_version in PINNED_DOWNLOAD_PACKAGES.items():
    try:
        installed_version = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        requirement = f"{distribution}=={expected_version}"
        if distribution == "huggingface-hub":
            requirement = f"huggingface_hub[hf_xet]=={expected_version}"
        requirements.append(requirement)

if requirements:
    print("Installing:", " ".join(requirements))
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-cache-dir",
            *requirements,
        ],
        check=True,
    )
else:
    print("All download dependencies already match.")

{
    name: importlib.metadata.version(name)
    for name in PINNED_DOWNLOAD_PACKAGES
}


## 3. Validate the Modal Volume and declare pinned sources

The ordinary notebook filesystem is ephemeral. This cell refuses to continue unless the expected `/mnt` path already exists and is writable. It does not create a similarly named directory outside an attached Volume, which could make a large download disappear when the kernel stops.


In [ ]:
import os
from pathlib import Path

VOLUME_ROOT = Path("/mnt/diffusion-arc-models")
if not VOLUME_ROOT.is_dir():
    raise RuntimeError(
        "The Modal Volume is not mounted at "
        f"{VOLUME_ROOT}. Attach the 'diffusion-arc-models' Volume in "
        "the notebook filesystem panel before continuing."
    )

write_probe = VOLUME_ROOT / ".diffusion-arc-write-probe"
try:
    write_probe.write_text("ok\n", encoding="utf-8")
    write_probe.unlink()
except OSError as error:
    raise RuntimeError(f"Modal Volume is not writable: {VOLUME_ROOT}") from error

FLUX_REPO_ID = "black-forest-labs/FLUX.2-klein-base-4B"
FLUX_REVISION = "8c44a2fbef88fae175da65df054db0f901aa9747"
STANDALONE_VAE_REPO_ID = "black-forest-labs/FLUX.2-dev"
STANDALONE_VAE_REVISION = "main"
NVARC_HANDLE = (
    "sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
)

FLUX_DIR = VOLUME_ROOT / "flux2-klein-base-4b"
STANDALONE_VAE_DIR = VOLUME_ROOT / "flux2-standalone-vae"
STANDALONE_VAE_PATH = STANDALONE_VAE_DIR / "ae.safetensors"
NVARC_DIR = VOLUME_ROOT / "nvarc-qwen3-4b-grids15-sft139"
MANIFEST_PATH = VOLUME_ROOT / "download-manifest.json"

EXPECTED_PROJECT_SHA256 = {
    "flux_transformer": (
        "9c5fed22b76baea749d88fc2abe3ad53245e7b21a0d353a762665eea00043b92"
    ),
    "diffusers_vae": (
        "ca70d2202afe6415bdbcb8793ba8cd99fd159cfe6192381504d6c4d3036e0f04"
    ),
    "standalone_vae": (
        "868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3"
    ),
}

EXPECTED_NVARC_FILE_SIZES = {
    "model-00001-of-00002.safetensors": 4_996_836_472,
    "model-00002-of-00002.safetensors": 2_270_397_024,
}

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

{
    "volume_root": str(VOLUME_ROOT),
    "flux": str(FLUX_DIR),
    "standalone_vae": str(STANDALONE_VAE_DIR),
    "nvarc": str(NVARC_DIR),
    "manifest": str(MANIFEST_PATH),
}


## 4. Check attached Modal Secrets

Modal injects attached Secret values as environment variables. This cell verifies only that the required names exist and never prints their values.


In [ ]:
def read_required_secret(name: str) -> str:
    value = os.environ.get(name)
    if not value or not value.strip():
        raise RuntimeError(
            f"Modal secret key {name!r} is missing or empty. Attach a "
            "Secret containing that exact key to this notebook and restart "
            "the kernel if necessary."
        )
    return value.strip()

SECRET_NAMES = ("HF_TOKEN", "KAGGLE_API_TOKEN")
for secret_name in SECRET_NAMES:
    os.environ[secret_name] = read_required_secret(secret_name)

print("Available Modal secret keys:", ", ".join(SECRET_NAMES))


## 5. Download the complete pinned FLUX snapshot

The complete repository layout supports both Diffusers-style component loading and the raw transformer file used by the project's strict inference path. The snapshot is pinned to the same revision as the repository's VAE smoke test. `snapshot_download` reuses matching files and can resume an interrupted transfer when this cell is rerun with the same destination. Expect approximately 23.7 GB.


In [ ]:
assert os.environ.get("HF_TOKEN"), "Check Modal Secrets first."

from huggingface_hub import snapshot_download

FLUX_DIR.mkdir(parents=True, exist_ok=True)
flux_snapshot_path = Path(
    snapshot_download(
        repo_id=FLUX_REPO_ID,
        revision=FLUX_REVISION,
        local_dir=str(FLUX_DIR),
        token=os.environ["HF_TOKEN"],
        force_download=False,
    )
)
assert flux_snapshot_path.resolve() == FLUX_DIR.resolve(), flux_snapshot_path
print("Complete FLUX snapshot:", flux_snapshot_path)


## 6. Download the standalone BFL VAE

The FLUX snapshot already contains a Diffusers-format VAE. Diffusion-ARC's strict vendored loader additionally uses the original BFL state-dict file named `ae.safetensors`. A 401 or 403 normally means that the `FLUX.2-dev` access conditions were not accepted for the account that owns `HF_TOKEN`.


In [ ]:
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError

STANDALONE_VAE_DIR.mkdir(parents=True, exist_ok=True)
try:
    downloaded_standalone_vae_path = Path(
        hf_hub_download(
            repo_id=STANDALONE_VAE_REPO_ID,
            filename="ae.safetensors",
            revision=STANDALONE_VAE_REVISION,
            local_dir=str(STANDALONE_VAE_DIR),
            token=os.environ["HF_TOKEN"],
            force_download=False,
        )
    )
except (GatedRepoError, HfHubHTTPError) as error:
    raise PermissionError(
        "Could not download the official standalone VAE. Accept access at "
        "https://huggingface.co/black-forest-labs/FLUX.2-dev and rerun "
        "this cell with a read token from the same account."
    ) from error

assert downloaded_standalone_vae_path.resolve() == (
    STANDALONE_VAE_PATH.resolve()
), downloaded_standalone_vae_path
print("Standalone BFL VAE:", downloaded_standalone_vae_path)


## 7. Download the pinned NVARC checkpoint from Kaggle

Kaggle version 1 contains two BF16 Safetensors shards plus model and tokenizer configuration. `output_dir` writes directly into the Modal Volume. Rerunning with `force_download=False` retains a complete existing download and allows KaggleHub to recover its cache state.


In [ ]:
assert os.environ.get("KAGGLE_API_TOKEN"), "Check Modal Secrets first."

import kagglehub

NVARC_DIR.mkdir(parents=True, exist_ok=True)
nvarc_download_path = Path(
    kagglehub.model_download(
        NVARC_HANDLE,
        output_dir=str(NVARC_DIR),
        force_download=False,
    )
)
print("NVARC checkpoint:", nvarc_download_path)


## 8. Flush writes and validate checkpoint structure

Modal Notebooks persist files written under an attached Volume. The explicit filesystem flush below is an additional boundary before validation. The checks fail early if a transfer ended before every required artifact arrived.


In [ ]:
if hasattr(os, "sync"):
    os.sync()

required_flux_entries = (
    "model_index.json",
    "scheduler",
    "text_encoder",
    "tokenizer",
    "transformer",
    "vae",
    "flux-2-klein-base-4b.safetensors",
)
missing_flux_entries = [
    name for name in required_flux_entries if not (FLUX_DIR / name).exists()
]
assert not missing_flux_entries, (
    f"Incomplete FLUX snapshot: {missing_flux_entries}"
)

assert STANDALONE_VAE_PATH.is_file(), STANDALONE_VAE_PATH

required_nvarc_files = (
    "added_tokens.json",
    "config.json",
    "generation_config.json",
    "model-00001-of-00002.safetensors",
    "model-00002-of-00002.safetensors",
    "model.safetensors.index.json",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json",
)
missing_nvarc_files = [
    name for name in required_nvarc_files if not (NVARC_DIR / name).is_file()
]
assert not missing_nvarc_files, (
    f"Incomplete NVARC checkpoint: {missing_nvarc_files}"
)

for filename, expected_size in EXPECTED_NVARC_FILE_SIZES.items():
    actual_size = (NVARC_DIR / filename).stat().st_size
    assert actual_size == expected_size, (
        filename,
        expected_size,
        actual_size,
    )

print("FLUX, both VAE formats, and NVARC have the required structure.")


## 9. Verify project checkpoint identities

These SHA-256 values are the artifact identities expected by Diffusion-ARC. Hashing reads roughly 8.3 GB from the Volume and can take several minutes, but prevents a similarly named or corrupted file from becoming experiment input.


In [ ]:
import hashlib

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

project_checkpoint_paths = {
    "flux_transformer": (
        FLUX_DIR / "flux-2-klein-base-4b.safetensors"
    ),
    "diffusers_vae": (
        FLUX_DIR / "vae" / "diffusion_pytorch_model.safetensors"
    ),
    "standalone_vae": STANDALONE_VAE_PATH,
}

actual_project_sha256 = {}
for name, path in project_checkpoint_paths.items():
    print(f"Hashing {name}: {path}")
    actual_project_sha256[name] = sha256_file(path)
    assert actual_project_sha256[name] == EXPECTED_PROJECT_SHA256[name], (
        name,
        EXPECTED_PROJECT_SHA256[name],
        actual_project_sha256[name],
    )

actual_project_sha256


## 10. Write a reproducibility manifest

The manifest records source identifiers, resolved Hugging Face revisions, persistent paths, sizes, hashes, and downloader versions. It contains no authentication material.


In [ ]:
import json
from datetime import datetime, timezone

from huggingface_hub import HfApi

def directory_size(path: Path) -> int:
    return sum(
        item.stat().st_size
        for item in path.rglob("*")
        if item.is_file()
    )

hf_api = HfApi(token=os.environ["HF_TOKEN"])
resolved_flux_revision = hf_api.model_info(
    FLUX_REPO_ID,
    revision=FLUX_REVISION,
).sha
resolved_vae_revision = hf_api.model_info(
    STANDALONE_VAE_REPO_ID,
    revision=STANDALONE_VAE_REVISION,
).sha

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "volume_root": str(VOLUME_ROOT),
    "downloaders": {
        name: importlib.metadata.version(name)
        for name in PINNED_DOWNLOAD_PACKAGES
    },
    "flux": {
        "repo_id": FLUX_REPO_ID,
        "requested_revision": FLUX_REVISION,
        "resolved_revision": resolved_flux_revision,
        "path": str(FLUX_DIR),
        "directory_bytes": directory_size(FLUX_DIR),
        "project_file_sha256": {
            name: actual_project_sha256[name]
            for name in ("flux_transformer", "diffusers_vae")
        },
    },
    "standalone_vae": {
        "repo_id": STANDALONE_VAE_REPO_ID,
        "requested_revision": STANDALONE_VAE_REVISION,
        "resolved_revision": resolved_vae_revision,
        "path": str(STANDALONE_VAE_PATH),
        "bytes": STANDALONE_VAE_PATH.stat().st_size,
        "sha256": actual_project_sha256["standalone_vae"],
    },
    "nvarc": {
        "kaggle_handle": NVARC_HANDLE,
        "version": 1,
        "path": str(NVARC_DIR),
        "directory_bytes": directory_size(NVARC_DIR),
        "weight_file_bytes": EXPECTED_NVARC_FILE_SIZES,
    },
}

temporary_manifest = MANIFEST_PATH.with_suffix(".json.tmp")
temporary_manifest.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)
temporary_manifest.replace(MANIFEST_PATH)
if hasattr(os, "sync"):
    os.sync()

print("Manifest written to:", MANIFEST_PATH)
manifest


## Paths for later loading and training

Attach the same Volume to future Modal Notebooks or Functions. The paths below remain stable as long as the Volume is mounted at `/mnt/diffusion-arc-models`.


In [ ]:
model_paths = {
    "diffusers_flux_snapshot": str(FLUX_DIR),
    "raw_flux_transformer": str(
        FLUX_DIR / "flux-2-klein-base-4b.safetensors"
    ),
    "diffusers_vae": str(FLUX_DIR / "vae"),
    "standalone_bfl_vae": str(STANDALONE_VAE_PATH),
    "nvarc": str(NVARC_DIR),
    "manifest": str(MANIFEST_PATH),
}

for name, path in model_paths.items():
    print(f"{name}: {path}")

model_paths


## Exercise: verify persistence after a kernel restart

Stop the CPU kernel after the successful download, start a fresh kernel with the same Volume attached, and rerun the dependency, mount/identifier, and path-check cells. Predict which paths would have disappeared if they had been placed under `/root` or `/tmp` instead of `/mnt/diffusion-arc-models`.

The answer scaffold below checks the durable paths without loading any model into RAM or GPU memory.


In [ ]:
persistent_path_check = {
    name: Path(path).exists()
    for name, path in model_paths.items()
}
assert all(persistent_path_check.values()), persistent_path_check
persistent_path_check


## Common pitfalls and optional extension

- **The mount check fails:** attach the `diffusion-arc-models` Volume in the notebook filesystem panel. Do not replace the check with `mkdir`, because that could create an ephemeral look-alike directory.
- **A Secret is missing:** attach a Modal Secret containing keys named exactly `HF_TOKEN` and `KAGGLE_API_TOKEN`, then restart the kernel if the environment does not update.
- **Hugging Face returns 401 or 403:** accept the `FLUX.2-dev` conditions while signed into the same account that owns the read token.
- **Kaggle authentication fails:** generate the current API token in Kaggle settings and store only its value in `KAGGLE_API_TOKEN`.
- **A download is interrupted:** retain the same Volume paths and rerun only the affected download cell. Avoid `force_download=True` unless intentionally replacing files.
- **A project hash fails:** do not train from that artifact. Rerun the affected download or investigate the upstream revision before changing an expected hash.
- **Credits disappear during downloading:** use a CPU kernel and switch to a GPU only after the durable files and manifest exist.
- **A future loader rejects the Diffusers VAE:** use `flux2-standalone-vae/ae.safetensors` for the strict vendored loader; the Diffusers VAE has different state-dict names.

**Optional extension:** create a separate GPU smoke-test notebook that mounts this Volume read-only and verifies one frozen NVARC forward pass plus FLUX/VAE checkpoint contracts. Keep that test separate from this download-only notebook so package and GPU requirements remain explicit.
